<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-sft-data.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Notebook for data preperation

Prepare a prompt-completion dataset for SFT.

- Prompt is a year with keywords, the task is to create a newspaper article
- Completion is an original newspaper article

We create a pseudo-task by extracting keywords from an article, and use this to create a prompt.

Most of this notebook is concerned with extracting keywords from articles and using this to reformat data into a prompt completion format.

In [8]:
import pandas as pd
from pathlib import Path
from ollama import Client
import json
from time import sleep
import asyncio
import json
import re
from ollama import AsyncClient
from datasets import Dataset

# Extract keyphrases with Ollama

In [2]:
# load a newspaper dataset
base_path = Path('/Volumes/X9 Pro/newspapers-csv/lwm-csv')
file_name = '0002610.csv'

In [3]:
df = pd.read_csv(base_path / file_name, index_col=0)
df['text_length'] = df['text'].str.len()
df.year.min(),df.year.max()

(np.int64(1879), np.int64(1920))

In [4]:
df_ocr = df[(df.ocr_quality_mean > 0.95) & (df.text_length > 1000)]

In [5]:
out_path = Path('data')
out_path.mkdir(exist_ok=True)
df_ocr_train = df_ocr.sample(frac=0.8, random_state=42)
df_ocr_test = df_ocr.drop(df_ocr_train.index)
df_ocr_train.to_csv(out_path / 'train.csv', index=False)
df_ocr_test.to_csv(out_path / 'test.csv', index=False)
df_ocr_train.shape, df_ocr_test.shape

((63202, 18), (15800, 18))

In [6]:
df_ocr_train_sample = df_ocr_train.sample(1000, random_state=42)
df_ocr_train_sample.to_csv(out_path / 'train_sample.csv', index=False)

In [12]:
# three prompt types
# guess the year of publication
# guess the year of publication with reasoning
# write a an article given the year of publication
# write an article given the year of publication with reasoning

In [17]:
with open('api_key.json') as f:
    api_key = json.load(f)['api_key']


import asyncio
import json
from ollama import AsyncClient

import asyncio
import json
from ollama import AsyncClient
from tqdm import tqdm


async def prompt_ollama(messages, client, model="llama3", max_retries=5):
    """Prompt the Ollama API with retries.
    
    Args:
        messages (list): A list of message dictionaries to send to the API.
        client (AsyncClient): An instance of the AsyncClient to use for making requests.
        model (str): The name of the model to use for the chat.
        max_retries (int): The maximum number of retry attempts in case of failure.
    Returns:
        str: The content of the response message from the API, or None if all retries fail"""

    for attempt in range(max_retries):
        try:
            response = await client.chat(
                model=model,
                messages=messages
            )
            #print(f"Response: {response}")
            return response["message"]["content"]

        except Exception as e:
            print(f"Retry {attempt+1}/{max_retries}: {e}")
            await asyncio.sleep(0.5)

    return None


async def process_messages(
    message_list : list,
    model : str ="llama3",
    outfile : str="results.jsonl",
    concurrency : int=10,
    ):
    """
    Process a list of messages by sending them to the Ollama API and saving the responses.
    Args:
        message_list (list): A list of message dictionaries to send to the API.
        model (str): The name of the model to use for the chat.
        outfile (str): The name of the output file to save the results in JSONL format.
        concurrency (int): The maximum number of concurrent API requests to make.

    Returns:
        None    
    """

    client = AsyncClient(host="https://ollama.com",
                        headers={'Authorization': api_key})
    semaphore = asyncio.Semaphore(concurrency)

    async def worker(messages):

        async with semaphore:

            response = await prompt_ollama(messages[1], client, model)

            record = {
                "context": messages[0],
                "messages": messages,
                "response": response
            }

            with open(outfile, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

            return record


    tasks = [worker(messages) for messages in message_list]

    with tqdm(total=len(tasks)) as pbar:

        for future in asyncio.as_completed(tasks):

            await future
            pbar.update(1)

In [18]:
import re
import unicodedata


def normalize_ocr_text(text: str, max_tokens: int = -1) -> str:
    """
    Clean and normalize OCR-style noisy text.

    Steps:
    - Normalize unicode
    - Remove hyphenated line breaks
    - Remove stray line breaks inside paragraphs
    - Collapse whitespace
    - Remove obvious OCR garbage characters
    """

    # Normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Fix hyphenated line breaks: "sup-\nplies" -> "supplies"
    text = re.sub(r"-\n\s*", "", text)

    # Replace remaining line breaks with spaces
    text = re.sub(r"\n+", " ", text)

    # Remove weird isolated punctuation artifacts
    text = re.sub(r"[•~\\]+", " ", text)

    # Remove stray quotes
    text = text.replace("’", "'").replace("`", "'")

    # Remove multiple punctuation artifacts
    text = re.sub(r"[\"']{2,}", "'", text)

    # Remove random standalone characters
    text = re.sub(r"\b[a-zA-Z]\b", "", text)

    # Collapse multiple spaces
    text = re.sub(r"\s{2,}", " ", text)

    # Max tokens (for LLM input limits)
    tokens = text.split()
    
    text = " ".join(tokens[:max_tokens])

    # Trim
    text = text.strip()

    return text

In [19]:
df_ocr_train_sample.columns

Index(['article_headline', 'item_type', 'ocr_quality_mean', 'ocr_quality_sd',
       'word_count', 'plain_text_file', 'date', 'newspaper_title', 'location',
       'source', 'text', 'year', 'month', 'day', 'NLP', 'issue', 'art_num',
       'text_length'],
      dtype='str')

In [20]:
messages_list = [
    [(row.name, row.year, row.text),
        (
            {"role": "system", "content": "You are a helpful editor creates an index by generative abtractive keyphrases from historical newspaper articles. The article is demarcated by triple hashtags. Abstractive key phrases are single words or longer phrases that describe the main topics, entities, and themes of the article. Return abstractive key phrases a JSON list of strings."},
            {"role": "user", "content": f"Extract ten key phrases from the following article: \n\n ### {normalize_ocr_text(row.text)[:1000]} ###. Only return abstractive key phrases a JSON list of strings."}
        )   
    ]
        for _, row in df_ocr_train_sample.iterrows()
]

In [21]:
messages_list[10]

[('0002610/1918/0615/0002610_19180615_art0129_metadata.xml',
  1918,
  "VOLUNTEER ORDERS.\n\n'Sunday (to-morrow).—Class; Firing at Altear.\nParade, 8.15 a.m., outside headquarters or Cen-\ntral Station, 8.40 a.m. Dress: Musketry Order;\none ration to be carried.\nMonday.-7.15 p.m., Visional Training; 8.15\n, p.m., Dayonet Fighting.\nThursclay.7-7.15 p.m. and 8.15 p.m., Entrench-\ning and Field Work.\nOrderly.—Corporal McDermid.\nP. WARBURTON,\nOfficer Commanding.\n\nMissionary Association.—The monthly meeting\nof the Missionary Association was held in the\nParish Room on Monday evening, and was pre-\nsided over by the Rev. J. Collins. The Vicar of\nAston (the Rev. W. L. Johnstone) gave a Most\ninteresting ii.ddress, dealing with missionary\nwork from the standpoint of home organisation.\nThe tragedy of the ehurch was, he said, the\nlarge percentage of Christian communicants who\ndo not believe in foreign missions. Until recent\nyears the cause of missions has occupied only a\nvery smal

In [ ]:
# prompt model and save results
await process_messages(messages_list, model="deepseek-v3.1:671b-cloud", outfile="key_phrases_1.jsonl", concurrency=5)

 98%|█████████▊| 982/1000 [26:15<00:37,  2.07s/it]

# Create training data

In [ ]:


def parse_llm_json(output: str):
    """
    Parse LLM output that may contain JSON either as plain text
    or inside ```json ... ``` code fences.

    Args:
        output (str): The text returned by the LLM.

    Returns:
        Parsed JSON as Python object (dict, list, etc.)
    """

    # Try to find ```json ... ``` block
    code_fence_match = re.search(r"```json\s*(.*?)```", output, re.DOTALL | re.IGNORECASE)
    
    if code_fence_match:
        json_str = code_fence_match.group(1).strip()
    else:
        # Fallback: assume the whole text is JSON
        json_str = output.strip()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Optional: fallback to safer literal_eval (only for lists/dicts)
        import ast
        try:
            return ast.literal_eval(json_str)
        except Exception as e:
            print(f"Failed to parse JSON from LLM output: {e}\nOutput was:\n{output}")
            return []
            #raise ValueError(f"Failed to parse JSON from LLM output: {e}\nOutput was:\n{output}")


In [9]:
# keyphrases = []
# with open("key_phrases_1.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         record = json.loads(line)
#         keyphrases.append(record)

keyphrases = []
with open("key_phrases_1.jsonl", "r", encoding="utf-8") as f:
    for i,line in enumerate(f):
        record = json.loads(line)
        out_dict = {}
        out_dict["year"] = str(record['context'][1])
        out_dict["text"] = str(record['context'][2])
        out_dict["response"] = str(parse_llm_json(record['response']))
        keyphrases.append(out_dict)

Failed to parse JSON from LLM output: unexpected character after line continuation character (<unknown>, line 1)
Output was:
\[
  "Mayor's power struggle",
  "Warrington town politics",
  "Council official authority",
  "Public rights assertion",
  "Mismanagement of power",
  "Transparency in local government",
  "Councillor oversight duties",
  "Municipal accountability",
  "Correction of errors",
  "Standing Order inspection rights"
\]
Failed to parse JSON from LLM output: invalid syntax (<unknown>, line 1)
Output was:
It looks like your request got cut off before the actual article text was provided after the hashtags. To generate abstractive keyphrases, I need the full content of the article.

Please provide the complete article text between the triple hashtags (###), and I will analyze its main topics, entities, and themes to create the requested JSON list of abstractive keyphrases.

For example, please format your input like this:

```
### [Full article text goes here] ###
```
Fa

In [10]:

dataset_keyphrases = Dataset.from_list(keyphrases)
dataset_keyphrases[0]

{'year': '1890',
 'text': "THE LATE MR. JOHN SAVAGE.\n\nThe funeral of the late Mr. John Savage took\nplace at the cemetery on Saturday afternoon. The\ndeceased, who was the third son of the late Mr. Wm.\nSavage, of Church-st., was 26 years of age at the\ntime of his death, which was due to typhoid fever.\nThe ceremony at the gravesidc was performed by\nthe Rev. Thomas Rigby, vicar of St. Peter's Church,\nof which the deceased was one of the sidesmen.\nThere was a large attendance. A number of choice\nwreaths had been forwarded.\nACCIDENT TO A WARRINGTON BOY AT CHESTER.\n\nAn accident happened, on Monday, to a lad named\nHesketh Parr, about seven years of age, who lives\nin Monks-st., Warrington. He had been with his\nparents to Chester with the picnic from St. Mary's\nChurch, Bank Quay, and when returning from\nChester he fell from the top of a 'bus to the ground,\nalighting on his head, which was badly injured. He\nwas immediately picked up and conveyed home by\ntrain. Ms wounds were

In [12]:

def preprocess_function(example):
    return {
        "prompt": [
                {"role": "user", "content": f"Generate an article published in {example['year']} given the key phrases {parse_llm_json(example['response'])}"}
                    ],
        "completion": [
            {"role": "assistant", "content": example['text']}
                    ],
    }

dataset = Dataset.from_list(keyphrases)
dataset = dataset.map(preprocess_function,remove_columns=['text', 'response', 'year'])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [15]:
dataset.push_to_hub("key_phrases_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Kaspar/key_phrases_dataset/commit/e995204c90dc98ff9ef4fadb5480412d61a649e8', commit_message='Upload dataset', commit_description='', oid='e995204c90dc98ff9ef4fadb5480412d61a649e8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kaspar/key_phrases_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kaspar/key_phrases_dataset'), pr_revision=None, pr_num=None)

# Fin